# Retention Radar — Exploratory Data Analysis

This notebook establishes the empirical retail facts across the two-year Dunnhumby Complete Journey dataset.
All computations import standardized logic from `src/retention_radar/eda/analysis.py`.

In [1]:
import duckdb

from retention_radar.config import get_project_root, load_project_config
from retention_radar.eda.analysis import (
    compute_churn_base_rate,
    compute_concentration,
    compute_demographic_coverage,
    compute_headline_kpis,
)

root = get_project_root()
cfg = load_project_config()
con = duckdb.connect(str(root / cfg.paths.warehouse_db), read_only=True)
print("Connected to DuckDB warehouse.")

Connected to DuckDB warehouse.


## 1. Headline Retail KPIs
Assessing customer panel volume, shopping frequency, and monetary throughput across 25 periods (days 1–700).

In [2]:
kpis = compute_headline_kpis(con)
for k, v in kpis.items():
    print(f"{k}: {v:,}" if isinstance(v, int) else f"{k}: {v}")
print(
    "\nFinding: The retailer tracks 2,500 frequent-shopper households across 272,110 baskets generating $7,921,184.70 in total sales, averaging 4.35 shopping trips per household per 28-day period with a mean basket size of $29.11."
)

total_households: 2,500
total_baskets: 272,110
total_revenue: 7921184.7
avg_basket_spend: 29.11
spend_per_household: 3168.47
trips_per_hh_per_period: 4.35

Finding: The retailer tracks 2,500 frequent-shopper households across 272,110 baskets generating $7,921,184.70 in total sales, averaging 4.35 shopping trips per household per 28-day period with a mean basket size of $29.11.


## 2. Revenue Concentration (Lorenz Curve & Gini)
Examining spend concentration across the customer base.

In [3]:
conc = compute_concentration(con)
print(f"Top 20% Revenue Share: {conc['top_20pct_revenue_share']}%")
print(f"Gini Coefficient: {conc['gini_coefficient']}")
print(
    "\nFinding: Customer spend exhibits strong Pareto concentration with a Gini coefficient of 0.5034; the top 20% of households generate 53.10% of total grocery revenue, justifying high-touch Champion retention programs."
)

Top 20% Revenue Share: 53.1%
Gini Coefficient: 0.5034

Finding: Customer spend exhibits strong Pareto concentration with a Gini coefficient of 0.5034; the top 20% of households generate 53.10% of total grocery revenue, justifying high-touch Champion retention programs.


## 3. Churn Base Rate & Empirical Retention Rhythm
Measuring empirical lapse probabilities for active shoppers.

In [4]:
churn = compute_churn_base_rate(con)
print(f"Active Household Occasions: {churn['total_active_instances']:,}")
print(f"Churn Instances (0 trips next 2 periods): {churn['churn_instances']:,}")
print(f"Empirical Churn Base Rate: {churn['churn_base_rate_pct']}%")
print(
    "\nFinding: The binary churn base rate across training cutoffs (periods 6 to 15) is 4.53% (896 churn events over 19,797 active observations), exceeding the 3% viable modeling floor without requiring artificial label re-sampling."
)

Active Household Occasions: 19,797
Churn Instances (0 trips next 2 periods): 896
Empirical Churn Base Rate: 4.53%

Finding: The binary churn base rate across training cutoffs (periods 6 to 15) is 4.53% (896 churn events over 19,797 active observations), exceeding the 3% viable modeling floor without requiring artificial label re-sampling.


## 4. Demographic Panel Coverage
Auditing demographic attribute availability and coverage.

In [5]:
demo = compute_demographic_coverage(con)
print(
    f"Households with Demographics: {demo['demographic_households']} / {demo['total_households']} ({demo['coverage_pct']}%)"
)
print(
    "\nFinding: Only 801 of 2,500 households (32.04%) have recorded demographic attributes, necessitating a mandatory has_demographics missingness indicator and fairness disparity audits for predictive models."
)

Households with Demographics: 801 / 2500 (32.04%)

Finding: Only 801 of 2,500 households (32.04%) have recorded demographic attributes, necessitating a mandatory has_demographics missingness indicator and fairness disparity audits for predictive models.
